# CS5811 — Phase 4: Train-Test Split & Export

**Phase 4 scope:** 1) load processed data  2) stratified 80/20 split with `random_state=42`  3) fit `StandardScaler` on train only  4) export shared splits for the team.

Every team member imports these files. Nobody re-cleans the data — that is what makes the Section 6 cross-member comparison fair.

## Setup

In [ ]:
import os, joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

os.makedirs('Files', exist_ok=True)

In [ ]:
df_enc = pd.read_csv('Files/processed_data.csv')
print(df_enc.shape)
df_enc.head()

(15000, 21)


,age,daily_screen_time_hours,phone_usage_before_sleep_minutes,sleep_duration_hours,sleep_quality_score,stress_level,caffeine_intake_cups,physical_activity_minutes,notifications_received_per_day,mental_fatigue_score,...,gender_Other,occupation_Doctor,occupation_Freelancer,occupation_Manager,occupation_Researcher,occupation_Software Engineer,occupation_Student,occupation_Teacher,stress_class,sleep_class
0,56,3.26,86,5.31,7.72,3.49,0,35,119,3.57,...,False,False,False,False,False,False,False,False,Medium,High
1,46,1.85,32,7.36,9.70,3.01,0,16,299,1.91,...,False,False,False,False,False,False,False,True,Medium,High
2,32,3.04,107,4.50,6.38,5.03,0,17,21,6.05,...,False,False,False,False,False,False,False,False,Medium,High
3,25,9.00,36,6.68,5.53,10.00,0,3,220,9.92,...,False,False,False,False,False,True,False,False,High,Medium
4,38,3.52,56,7.57,6.69,6.71,4,92,167,5.99,...,False,False,False,False,False,False,False,True,High,High


---
# Phase 4 — Train-Test Split & Export

### 1 · Separate features and targets

In [ ]:
target_cols  = ['stress_level', 'sleep_quality_score', 'stress_class', 'sleep_class']
feature_cols = [c for c in df_enc.columns if c not in target_cols]

X = df_enc[feature_cols].astype(float)

y_stress_reg = df_enc['stress_level']
y_stress_cls = df_enc['stress_class']
y_sleep_reg  = df_enc['sleep_quality_score']
y_sleep_cls  = df_enc['sleep_class']

print('X shape:', X.shape)

X shape: (15000, 17)


### 2 · Stratified 80/20 split

In [ ]:
idx = np.arange(len(X))
train_idx, test_idx = train_test_split(
    idx, test_size=0.20,
    stratify=y_stress_cls,
    random_state=RANDOM_STATE
)

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]

y_train_stress_reg = y_stress_reg.iloc[train_idx]; y_test_stress_reg = y_stress_reg.iloc[test_idx]
y_train_stress_cls = y_stress_cls.iloc[train_idx]; y_test_stress_cls = y_stress_cls.iloc[test_idx]
y_train_sleep_reg  = y_sleep_reg.iloc[train_idx];  y_test_sleep_reg  = y_sleep_reg.iloc[test_idx]
y_train_sleep_cls  = y_sleep_cls.iloc[train_idx];  y_test_sleep_cls  = y_sleep_cls.iloc[test_idx]

print('train:', X_train.shape, '| test:', X_test.shape)
print('\ntrain class ratios:\n', y_train_stress_cls.value_counts(normalize=True).round(3))

train: (12000, 17) | test: (3000, 17)

train class ratios:
 stress_class
High      0.632
Medium    0.264
Low       0.103
Name: proportion, dtype: float64


Stratification on `stress_class` keeps Low / Medium / High proportions identical between train and test.

### 3 · Scale features (fit on train only)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns, index=X_train.index)
X_test_scaled  = pd.DataFrame(X_test_scaled,  columns=X.columns, index=X_test.index)

`StandardScaler` is required for SVM, k-NN, MLP and DNN where distances and gradients are sensitive to feature magnitude. Tree-based methods (Random Forest, Decision Tree) and regularised Logistic Regression do not need scaling, so tree-based members import the raw splits directly.

### 4 · Export shared splits

In [ ]:
X_train.to_csv('Files/X_train.csv', index=False)
X_test.to_csv('Files/X_test.csv',   index=False)

X_train_scaled.to_csv('Files/X_train_scaled.csv', index=False)
X_test_scaled.to_csv('Files/X_test_scaled.csv',   index=False)

pd.DataFrame({
    'stress_reg': y_train_stress_reg.values,
    'stress_cls': y_train_stress_cls.values,
    'sleep_reg':  y_train_sleep_reg.values,
    'sleep_cls':  y_train_sleep_cls.values
}).to_csv('Files/y_train.csv', index=False)

pd.DataFrame({
    'stress_reg': y_test_stress_reg.values,
    'stress_cls': y_test_stress_cls.values,
    'sleep_reg':  y_test_sleep_reg.values,
    'sleep_cls':  y_test_sleep_cls.values
}).to_csv('Files/y_test.csv', index=False)

joblib.dump(scaler, 'Files/scaler.pkl')

print('Saved:')
for f in sorted(os.listdir('Files')):
    print(' -', f)

Saved:
 - X_test.csv
 - X_test_scaled.csv
 - X_train.csv
 - X_train_scaled.csv
 - feature_names.csv
 - label_encoder.pkl
 - processed_data.csv
 - raw_inspected.csv
 - scaler.pkl
 - y_test.csv
 - y_train.csv


## Deliverables

```
Files/
├── raw_inspected.csv      (Phase 1)
├── processed_data.csv     (Phase 3)
├── X_train.csv            unscaled (RF, Decision Tree, regularised LogReg)
├── X_test.csv
├── X_train_scaled.csv     scaled (SVM, k-NN, MLP, DNN, CNN)
├── X_test_scaled.csv
├── y_train.csv            4 target views: stress_reg, stress_cls, sleep_reg, sleep_cls
├── y_test.csv
├── scaler.pkl             fitted StandardScaler (joblib.load)
├── label_encoder.pkl      (Phase 3)
└── feature_names.csv      (Phase 3)
```

Next: **Phase 5 — Machine Learning Prediction** (Random Forest + Decision Tree).